In [1]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from datetime import datetime

batch_id = datetime.now().strftime("%Y%m%d%H%M%S")

for schema_name in ["bronze", "silver", "gold", "quarantine"]:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {schema_name}")

print(f"Starting SCD Type 2 build. Batch ID: {batch_id}")


StatementMeta(, 55f18dcc-2b9c-4b0e-8700-58337aafe374, 3, Finished, Available, Finished, False)

Starting SCD Type 2 build. Batch ID: 20260714022744


In [2]:
def write_delta(df, table_name):
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(table_name)
    )
    print(f"{table_name}: {df.count()} rows")


def add_hash_column(df, hash_columns, hash_col_name="_record_hash"):
    """
    Creates a hash value from important descriptive columns.
    If any of these columns change, the hash changes.
    """
    hash_expr = F.sha2(
        F.concat_ws(
            "||",
            *[
                F.coalesce(F.col(c).cast("string"), F.lit("__NULL__"))
                for c in hash_columns
            ]
        ),
        256
    )

    return df.withColumn(hash_col_name, hash_expr)

StatementMeta(, 55f18dcc-2b9c-4b0e-8700-58337aafe374, 4, Finished, Available, Finished, False)

In [3]:
customer_hash_columns = [
    "first_name",
    "last_name",
    "email",
    "phone",
    "city",
    "province",
    "country",
    "customer_segment",
    "loyalty_status"
]

customers = spark.table("silver.customers")

customers_base = (
    customers
    .select(
        "customer_id",
        "first_name",
        "last_name",
        "email",
        "phone",
        "city",
        "province",
        "country",
        "customer_segment",
        "loyalty_status",
        "created_at",
        "updated_at",
        "change_type",
        "effective_date",
        "_source_file_name",
        "_ingestion_timestamp",
        "_batch_id"
    )
    .withColumn("effective_date", F.to_date("effective_date"))
)

customers_hashed = add_hash_column(
    customers_base,
    customer_hash_columns,
    "_customer_hash"
)

customer_window = Window.partitionBy("customer_id").orderBy("effective_date", "_ingestion_timestamp")

customers_with_lag = (
    customers_hashed
    .withColumn("_previous_hash", F.lag("_customer_hash").over(customer_window))
)

# Keep first version and only real changes after that
customers_changed_only = (
    customers_with_lag
    .filter(
        F.col("_previous_hash").isNull() |
        (F.col("_customer_hash") != F.col("_previous_hash"))
    )
)

customer_scd_window = Window.partitionBy("customer_id").orderBy("effective_date")

dim_customer = (
    customers_changed_only
    .withColumn("effective_start_date", F.col("effective_date"))
    .withColumn(
        "effective_end_date",
        F.date_sub(F.lead("effective_date").over(customer_scd_window), 1)
    )
    .withColumn(
        "effective_end_date",
        F.coalesce(F.col("effective_end_date"), F.to_date(F.lit("9999-12-31")))
    )
    .withColumn(
        "is_current",
        F.when(F.col("effective_end_date") == F.to_date(F.lit("9999-12-31")), F.lit(True))
         .otherwise(F.lit(False))
    )
)

customer_sk_window = Window.orderBy("customer_id", "effective_start_date")

dim_customer = (
    dim_customer
    .withColumn("customer_sk", F.row_number().over(customer_sk_window))
    .select(
        "customer_sk",
        "customer_id",
        "first_name",
        "last_name",
        "email",
        "phone",
        "city",
        "province",
        "country",
        "customer_segment",
        "loyalty_status",
        "created_at",
        "updated_at",
        "change_type",
        "effective_start_date",
        "effective_end_date",
        "is_current",
        "_customer_hash",
        "_source_file_name",
        "_ingestion_timestamp",
        "_batch_id"
    )
)

write_delta(dim_customer, "gold.dim_customer")

StatementMeta(, 55f18dcc-2b9c-4b0e-8700-58337aafe374, 5, Finished, Available, Finished, False)

gold.dim_customer: 127 rows


In [4]:
product_hash_columns = [
    "product_name",
    "category",
    "brand",
    "supplier_id",
    "unit_price",
    "cost_price",
    "is_active"
]

products = spark.table("silver.products")

products_base = (
    products
    .select(
        "product_id",
        "product_name",
        "category",
        "brand",
        "supplier_id",
        "unit_price",
        "cost_price",
        "is_active",
        "created_at",
        "updated_at",
        "change_type",
        "effective_date",
        "_source_file_name",
        "_ingestion_timestamp",
        "_batch_id"
    )
    .withColumn("effective_date", F.to_date("effective_date"))
)

products_hashed = add_hash_column(
    products_base,
    product_hash_columns,
    "_product_hash"
)

product_window = Window.partitionBy("product_id").orderBy("effective_date", "_ingestion_timestamp")

products_with_lag = (
    products_hashed
    .withColumn("_previous_hash", F.lag("_product_hash").over(product_window))
)

# Keep first version and only real changes after that
products_changed_only = (
    products_with_lag
    .filter(
        F.col("_previous_hash").isNull() |
        (F.col("_product_hash") != F.col("_previous_hash"))
    )
)

product_scd_window = Window.partitionBy("product_id").orderBy("effective_date")

dim_product = (
    products_changed_only
    .withColumn("effective_start_date", F.col("effective_date"))
    .withColumn(
        "effective_end_date",
        F.date_sub(F.lead("effective_date").over(product_scd_window), 1)
    )
    .withColumn(
        "effective_end_date",
        F.coalesce(F.col("effective_end_date"), F.to_date(F.lit("9999-12-31")))
    )
    .withColumn(
        "is_current",
        F.when(F.col("effective_end_date") == F.to_date(F.lit("9999-12-31")), F.lit(True))
         .otherwise(F.lit(False))
    )
)

product_sk_window = Window.orderBy("product_id", "effective_start_date")

dim_product = (
    dim_product
    .withColumn("product_sk", F.row_number().over(product_sk_window))
    .select(
        "product_sk",
        "product_id",
        "product_name",
        "category",
        "brand",
        "supplier_id",
        "unit_price",
        "cost_price",
        "is_active",
        "created_at",
        "updated_at",
        "change_type",
        "effective_start_date",
        "effective_end_date",
        "is_current",
        "_product_hash",
        "_source_file_name",
        "_ingestion_timestamp",
        "_batch_id"
    )
)

write_delta(dim_product, "gold.dim_product")

StatementMeta(, 55f18dcc-2b9c-4b0e-8700-58337aafe374, 6, Finished, Available, Finished, False)

gold.dim_product: 36 rows


In [5]:
display(
    spark.table("gold.dim_customer")
    .groupBy("customer_id")
    .count()
    .filter(F.col("count") > 1)
    .orderBy("customer_id")
)

StatementMeta(, 55f18dcc-2b9c-4b0e-8700-58337aafe374, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 4aa7603e-bc9c-4484-a09e-39f2ed319cb0)

In [6]:
display(
    spark.table("gold.dim_customer")
    .filter(F.col("customer_id").isin("C00005", "C00017", "C00035", "C00072", "C00101"))
    .orderBy("customer_id", "effective_start_date")
)

StatementMeta(, 55f18dcc-2b9c-4b0e-8700-58337aafe374, 8, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 1e6ff890-c71d-4e6c-afb0-73de7764da1c)

In [7]:
display(
    spark.table("gold.dim_product")
    .groupBy("product_id")
    .count()
    .filter(F.col("count") > 1)
    .orderBy("product_id")
)

StatementMeta(, 55f18dcc-2b9c-4b0e-8700-58337aafe374, 9, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, f6d9ebf0-0165-4e1c-af07-dabc12f7b32c)

In [8]:
display(
    spark.table("gold.dim_product")
    .filter(F.col("product_id").isin("P00002", "P00009", "P00016", "P00025", "P00027"))
    .orderBy("product_id", "effective_start_date")
)

StatementMeta(, 55f18dcc-2b9c-4b0e-8700-58337aafe374, 10, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, c001fc03-cc45-4d4e-9a11-a0eeba4cdc87)

In [9]:
display(
    spark.table("gold.dim_customer")
    .filter(F.col("is_current") == True)
    .limit(20)
)

StatementMeta(, 55f18dcc-2b9c-4b0e-8700-58337aafe374, 11, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 35ca13be-fc25-409d-b13e-995c955c7665)

In [10]:
display(
    spark.table("gold.dim_product")
    .filter(F.col("is_current") == True)
    .limit(20)
)

StatementMeta(, 55f18dcc-2b9c-4b0e-8700-58337aafe374, 12, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 0feeaba8-580d-4cfa-8841-2f32a29e89a5)